# Data Modeling Notebook

## 1. Imports

Import the core libraries used for data loading, preprocessing, modeling, and evaluation.

In [1]:
import os
import polars as pl
import pandas as pd
import numpy as np
import tensorflow as tf

# Split, evaluation
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix

# Models
from sklearn.ensemble import AdaBoostClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from tensorflow.keras import layers, models

---

## 2. Load the cleaned dataset

Load the preprocessed IoT traffic dataset that was generated in the previous notebook. This dataset will be used as the input for all subsequent modeling steps.

In [2]:
# set the filepath to the parquet cleaned dataset
PATH = "../Data/Cleaned/Cleaned_Dataset.parquet"

# scan the parquet file with polars this gives us a lazy dataframe that we can manipulate without loading the entire dataset into memory
lazy_frame = pl.scan_parquet(PATH)

---

## 3. Define features and target

Select the input features **X** and the target labels **y** that we want the models to learn.  
The target variable encodes the different IoT traffic classes (for example: BENIGN, MIRAI, SPOOFING, RECON, etc.).

In [3]:
# Target column
TARGET_COL = "Label_Family"    

# Columns that should NEVER be used as features
label_like_cols = ["Label", "Label_Family", "Label_Binary"]

# Work out which columns are features
schema = lazy_frame.collect_schema()
all_cols = list(schema.keys())

# Exclude all label columns from the feature set
feature_cols = [c for c in all_cols if c not in label_like_cols]

In [4]:
# Collect just the columns we need into memory
df = lazy_frame.select(feature_cols + label_like_cols).collect()

### 3.1 Class-balanced sampling

Because the dataset is imbalanced across classes, we create a class-balanced sample for training.  
This helps the models see a more even distribution of classes during training and can improve performance on minority classes.

In [5]:
# Number of rows per category 10,000
N = 200000

# shuffle first so head(N) is effectively a random sample per class
df_shuffled = df.sample(fraction=1.0, with_replacement=False, seed=42)

df_balanced = (
    df_shuffled
        .group_by("Label_Family")
        .head(N)
)

In [6]:
# Convert to NumPy for the models
X = df_balanced.select(feature_cols).to_numpy()
y = df_balanced.get_column(TARGET_COL).to_numpy()

In [7]:
# Encoding the target variable because it is categorical
le = LabelEncoder()
y_encoded = le.fit_transform(df_balanced.get_column("Label_Family").to_numpy())
num_classes = len(le.classes_)

---

## 4. Train / test split

Split the dataset into training and test sets.  
The training set is used to fit the models, and the test set is held out for final evaluation of generalization performance.

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded,
    test_size=0.2,
    stratify=y_encoded,
    random_state=42
)

---

## 5. Model Selection Rationale

In this section we compare three tree-based ensembles (AdaBoost, LightGBM, CatBoost) and three deep learning models (CNN, LSTM, RNN) on the CIC-IoT 2023 dataset to identify the most effective approach for multiclass IoT intrusion detection.


### Evaluation Function for Tree-Based Models

This function is used to compute and display evaluation metrics (such as accuracy, precision, recall, F1-score, and confusion matrices) for each trained model.

In [ ]:
def eval_model(name, model):
    y_pred = model.predict(X_test)

    # Only suppress zero-division warnings for AdaBoost    
    zero_division_value = 0 if name == "AdaBoost" else "warn"

    print(f"\n===== {name} =====")
    print(classification_report(
        y_test,
        y_pred,
        digits=4,
        zero_division=zero_division_value,
    ))
    print(confusion_matrix(y_test, y_pred))

### 5.1 AdaBoost

Limited to 200 estimators, chosen for consistency with classical ensemble configurations.

In [10]:
ada = AdaBoostClassifier(
    n_estimators=200,
    learning_rate=0.5,
    random_state=42
)

ada.fit(X_train, y_train);

### 5.2 LightGBM

Tuned for multi-class classification with 500 estimators, moderate learning rate, and controlled leaf growth to reduce overfitting.

In [11]:
lgbm = LGBMClassifier(
    objective="multiclass",
    num_class=num_classes,
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=64,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbosity=-1,    
    device_type="gpu"   
)

lgbm.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    eval_metric="multi_logloss"
);

### 5.3 CatBoost

Trained using its built-in multiclass loss with default handling of categorical and numerical features.

In [12]:
cat = CatBoostClassifier(
    loss_function="MultiClass",
    iterations=500,
    learning_rate=0.1,
    depth=6,
    random_seed=42,    
    task_type="GPU",
    devices="0",
    silent=True,
)

cat.fit(
    X_train, y_train,
    eval_set=(X_test, y_test),
);

### 5.4 Prediction and Evaluation Workflow

Fit each model on the training data, generate predictions on the test set, and compute the evaluation metrics using the helper functions defined earlier.

In [24]:
eval_model("AdaBoost", ada)
eval_model("LightGBM", lgbm)
eval_model("CatBoost", cat)


===== AdaBoost =====
              precision    recall  f1-score   support

           0     0.3154    0.0118    0.0227     40000
           1     0.0000    0.0000    0.0000      2504
           2     0.5472    0.5762    0.5613     40000
           3     0.5660    0.4999    0.5309     40000
           4     0.9322    0.9910    0.9607     40000
           5     0.4116    0.8772    0.5603     40000
           6     0.6801    0.6895    0.6847     40000
           7     0.0000    0.0000    0.0000      4742

    accuracy                         0.5898    247246
   macro avg     0.4316    0.4557    0.4151    247246
weighted avg     0.5586    0.5898    0.5372    247246



C:\Users\Gerard Corrales\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\Gerard Corrales\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\Gerard Corrales\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages


===== LightGBM =====
              precision    recall  f1-score   support

           0     0.7750    0.8361    0.8044     40000
           1     0.7916    0.3686    0.5030      2504
           2     0.8499    0.7436    0.7932     40000
           3     0.7721    0.8690    0.8177     40000
           4     0.9991    0.9987    0.9989     40000
           5     0.7615    0.8171    0.7883     40000
           6     0.9304    0.8881    0.9088     40000
           7     0.7763    0.2992    0.4320      4742

    accuracy                         0.8431    247246
   macro avg     0.8320    0.7275    0.7558    247246
weighted avg     0.8461    0.8431    0.8403    247246


===== CatBoost =====
              precision    recall  f1-score   support

           0     0.7527    0.8269    0.7881     40000
           1     0.8167    0.3131    0.4527      2504
           2     0.8485    0.7295    0.7845     40000
           3     0.7630    0.8700    0.8130     40000
           4     0.9986    0.9986 

### **AdaBoost Interpretation**

- AdaBoost performs poorly on this dataset, with an overall accuracy of **58.98%**.
- The model completely fails to classify **class 1 (DOS)** and **class 7 (WEB)**, producing zero precision and recall for both.
- Only **class 4 (BENIGN)** achieves a strong F1-score, while classes 2, 3, 5, and 6 show moderate but inconsistent performance.
- The macro-average F1-score of **0.41** indicates that AdaBoost does not generalize across classes.
- These results show that AdaBoost is not suitable for this multi-class intrusion detection problem.

### **LightGBM Interpretation**

- LightGBM delivers the strongest performance among all models, achieving **84.31% accuracy**.
- The model performs exceptionally well on major attack families, with high F1-scores for classes **0, 2, 3, 4, 5, and 6**.
- Class **4 (BENIGN)** is almost perfectly classified with an F1-score near **1.0**.
- Lower recall values for **class 1 (DOS)** and **class 7 (WEB)** reflect difficulty detecting minority or overlapping traffic patterns.
- LightGBM provides the most balanced and reliable results and is the best-performing model in this evaluation.

### **CatBoost Interpretation**

- CatBoost achieves strong results with an overall accuracy of **83.32%**, close to LightGBM.
- The model performs well on classes **0, 2, 3, 4, 5, and 6**, showing good pattern separation.
- Similar to LightGBM, CatBoost struggles with **class 1 (DOS)** and **class 7 (WEB)**, both having low recall values.
- While competitive, CatBoost is slightly less consistent than LightGBM, especially in handling minority families.
- CatBoost remains a strong alternative but does not surpass the balanced performance of LightGBM.

### 5.5 Deep Learning Setup
Each sample is reshaped to
**(timesteps, features)** so that sequence-based layers can be applied consistently across models.

In [14]:
# Reuse existing split: X_train, X_test, y_train, y_test
input_dim = X_train.shape[1]
num_classes = len(np.unique(y_train))

# Reshape features for sequence-based models: (samples, timesteps, channels)
X_train_seq = X_train.reshape(-1, input_dim, 1)
X_test_seq  = X_test.reshape(-1, input_dim, 1)

### Evaluation Function for Deep Learning Models

This function is used to compute and display evaluation metrics (such as accuracy, precision, recall, F1-score, and confusion matrices) for each trained model.

In [15]:
def eval_dl_model(name, model, X, y_true):
    y_proba = model.predict(X)
    y_pred  = np.argmax(y_proba, axis=1)

    # Suppress zero-division warnings only for CNN and RNN
    zero_division_value = 0 if name in ["CNN", "RNN"] else "warn"

    print(f"\n=== {name} ===")
    print(classification_report(y_true, y_pred,
                                zero_division=zero_division_value))
    print(confusion_matrix(y_true, y_pred))

#### 5.5.1 Convolutional Neural Network (CNN)

This model applies 1D convolutions over the feature dimension to learn local patterns between related traffic
statistics, followed by dense layers and dropout for regularization.

In [18]:
cnn = models.Sequential([
    layers.Input(shape=(input_dim, 1)),
    layers.Conv1D(32, kernel_size=3, activation="relu"),
    layers.MaxPooling1D(pool_size=2),
    layers.Conv1D(64, kernel_size=3, activation="relu"),
    layers.GlobalAveragePooling1D(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(num_classes, activation="softmax"),
])

cnn.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

history_cnn = cnn.fit(
    X_train_seq,
    y_train,
    validation_split=0.2,
    epochs=10,
    batch_size=256,
    verbose=0,
)

#### 5.5.2 Long Short-Term Memory (LSTM)

The LSTM model treats each sample as an ordered sequence of features, allowing it to capture longer-range
dependencies that may arise in multi-step IoT attack behavior.


In [19]:
lstm = models.Sequential([
    layers.Input(shape=(input_dim, 1)),
    layers.LSTM(64, return_sequences=False),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(num_classes, activation="softmax"),
])

lstm.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

history_lstm = lstm.fit(
    X_train_seq,
    y_train,
    validation_split=0.2,
    epochs=10,
    batch_size=256,
    verbose=0,
)

#### 5.5.3 Recurrent Neural Network (RNN)

The vanilla RNN serves as a simpler recurrent baseline to compare against LSTMs. It still models the feature
vector as a short sequence but lacks the gating mechanisms of LSTMs, making it useful as a baseline
sequence model.


In [20]:
rnn = models.Sequential([
    layers.Input(shape=(input_dim, 1)),
    layers.SimpleRNN(64, return_sequences=False),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(num_classes, activation="softmax"),
])

rnn.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

history_rnn = rnn.fit(
    X_train_seq,
    y_train,
    validation_split=0.2,
    epochs=10,
    batch_size=256,
    verbose=0,
)

In [21]:
eval_dl_model("CNN", cnn, X_test_seq, y_test)
eval_dl_model("LSTM", lstm, X_test_seq, y_test)
eval_dl_model("RNN", rnn, X_test_seq, y_test)

7727/7727 ━━━━━━━━━━━━━━━━━━━━ 12s 2ms/step

=== CNN ===
              precision    recall  f1-score   support

           0       0.61      0.67      0.64     40000
           1       0.00      0.00      0.00      2504
           2       0.92      0.31      0.46     40000
           3       0.58      0.98      0.73     40000
           4       1.00      0.98      0.99     40000
           5       0.65      0.71      0.68     40000
           6       0.66      0.65      0.66     40000
           7       0.10      0.00      0.00      4742

    accuracy                           0.70    247246
   macro avg       0.57      0.54      0.52    247246
weighted avg       0.72      0.70      0.67    247246

[[26847     0     0     0     0  6045  7102     6]
 [  700     0     0     0     0  1377   427     0]
 [    0     0 12446 27512    26     5    11     0]
 [    0     0   971 39011     9     1     8     0]
 [    1     0   142   456 39375     0    26     0]
 [ 7121     0     0     0     0 28584

---

## 6. Results and Interpretation

### 6.1 Convolutional Neural Network (CNN) Interpretation

**Key results (CNN)**  
- Overall accuracy: **0.67**  
- Macro-average F1-score: **0.49**  
- Weighted-average F1-score: **0.63**

**Per-class performance (CNN)**  
- The CNN achieves moderate performance for several classes:
  - Class 0: precision 0.59, recall 0.65, F1 = 0.62  
  - Class 2: precision 0.53, recall 0.98, F1 = 0.69 (very high recall but relatively low precision)  
  - Class 4: precision 1.00, recall 0.98, F1 = 0.99 (nearly perfect classification)  
  - Classes 5 and 6: F1-scores around 0.67 and 0.68, indicating reasonable detection quality.
- The model performs poorly on:
  - Class 3: recall drops to 0.15 (most true class-3 samples are misclassified, often as class 2).  
  - Classes 1 and 7: precision, recall, and F1-score are all 0.00, meaning the CNN does not predict these classes at all.

**Confusion patterns (CNN)**  
- The confusion matrix shows that:
  - A large portion of class 2 is predicted correctly, but some samples are confused with classes 3 and 4.
  - A substantial share of class 3 samples is misclassified as class 2, which explains the very low recall for class 3.
  - Many class 0 samples are misclassified as classes 5 and 6.
  - WEB traffic (class 7) is never predicted; its samples are spread across other classes, mainly 5 and 6.

**Interpretation (CNN)**  
Overall, the CNN reaches a moderate accuracy but shows clear instability across classes. It tends to favor a subset of dominant patterns (especially classes 2, 4, 5, and 6) while completely ignoring minority classes such as DOS (class 1) and WEB (class 7). This suggests that the CNN architecture, as configured here, is not sufficiently capturing the temporal or structural patterns needed to separate all attack families, and it is especially weak for rare or overlapping traffic types.

### 6.2 Long Short-Term Memory Network (LSTM) Interpretation

**Key results (LSTM)**  
- Overall accuracy: **0.80**  
- Macro-average F1-score: **0.66**  
- Weighted-average F1-score: **0.79**

**Per-class performance (LSTM)**  
- The LSTM clearly outperforms the CNN and the simple RNN:
  - Class 0: F1 = 0.74, indicating good detection of this family.  
  - Class 2: F1 = 0.74, with a balanced trade-off between precision (0.87) and recall (0.64).  
  - Class 3: F1 = 0.80, with high recall (0.91), meaning most class-3 attacks are correctly identified.  
  - Class 4: F1 = 1.00, reflecting almost perfect detection of BENIGN traffic.  
  - Classes 5 and 6: F1-scores of 0.75 and 0.85, showing strong performance on these attack families.
- The weakest classes remain:
  - Class 1 (DOS): precision 0.99 but recall 0.19, giving an F1-score of 0.32. The model is very conservative when predicting DOS; when it predicts DOS it is almost always correct, but it misses most true DOS samples.  
  - Class 7 (WEB): precision 0.79 but recall 0.06, with an F1-score of 0.12. This indicates that WEB traffic is rarely detected and is heavily confused with other classes.

**Confusion patterns (LSTM)**  
- The confusion matrix shows:
  - Many class 2 samples are still misclassified as class 3 (and vice versa), which reflects some overlap between these two families.  
  - Class 3 has strong recall, with most samples correctly assigned to the right class.  
  - For class 1, many true DOS samples are classified as other attack types (especially class 5 or 6), which explains the low recall.  
  - WEB samples are mostly scattered across classes 5 and 6, with only a small fraction correctly identified as class 7.

**Interpretation (LSTM)**  
The LSTM delivers the best results among the deep learning models. It captures temporal patterns in the flows more effectively than the CNN and the simple RNN, leading to higher accuracy and more balanced performance across the main attack families. However, DOS and WEB remain challenging classes, with low recall due to class imbalance and feature overlap. Even so, the LSTM demonstrates that sequence-based modeling is a strong approach for this intrusion detection task and can rival tree-based methods on several families.

### 6.3 Simple Recurrent Neural Network (RNN) Interpretation

**Key results (RNN)**  
- Overall accuracy: **0.56**  
- Macro-average F1-score: **0.41**  
- Weighted-average F1-score: **0.53**

**Per-class performance (RNN)**  
- The RNN performs clearly worse than the LSTM and also worse than the CNN:
  - Class 2: F1 = 0.63 and class 3: F1 = 0.56, showing only moderate detection quality.  
  - Class 4: F1 = 0.94, indicating the model can still reliably identify BENIGN traffic.  
  - Class 5: F1 = 0.54 and class 6: F1 = 0.49, which reflects unstable and inconsistent predictions.
- The weakest classes are:
  - Class 0: F1 = 0.12, with recall of only 0.06. Most class-0 samples are misclassified as other families.  
  - Class 1 (DOS): F1 = 0.00, with zero recall, meaning the model never predicts this class.  
  - Class 7 (WEB): F1 = 0.00, again indicating that WEB traffic is never predicted.

**Confusion patterns (RNN)**  
- The confusion matrix reveals heavy misclassification:
  - Many class 0 samples are labeled as classes 5 and 6.  
  - DOS and WEB samples are almost entirely absorbed into other classes, especially 5 and 6.  
  - For several classes, a large portion of the true samples is redistributed across a small number of dominant predictions, which indicates that the RNN collapses to a few frequent labels.

**Interpretation (RNN)**  
The simple RNN struggles to model the complex temporal structure of the network flows and is strongly affected by class imbalance. Compared with the LSTM, it fails to retain useful longer-term dependencies and tends to collapse minority classes into a few dominant predictions. The overall accuracy and macro-average F1-score are low, and the complete failure to detect DOS and WEB suggests that the RNN architecture is not adequate for this multiclass IoT intrusion detection problem in its current form.

### 6.4 Deep Learning Models – Summary

Across the three deep learning architectures, the LSTM provides the most competitive results, achieving 80 percent accuracy and a macro-average F1-score of 0.66. The CNN shows moderate performance, while the simple RNN performs poorly and fails to detect several classes. All deep learning models have difficulty identifying the WEB family and, to a lesser extent, the DOS family. These findings indicate that sequence-based models such as LSTMs are promising for this task, but additional work on handling class imbalance, improving feature representations, or using more advanced architectures (such as attention-based models or 1D CNN–LSTM hybrids) would be needed to close the gap with the best tree-based methods and to improve detection of rare attack families.

### 6.5 Overall Model Comparison and Selected Model


- LightGBM: best overall accuracy and macro-F1, decent behavior on most classes but still struggles on DOS/WEB.

- CatBoost: close second, slightly worse on minority classes.

- AdaBoost: clearly underperforms, especially on DOS and WEB.

- Deep Learning models: CNN/LSTM/RNN performance vs tree-based models (e.g., trees > DL here).

- **Final choice of “best model”** for the project (likely LightGBM).

---